This notebook aggregate Poisson HMC and Neutra Chains together and conduct diagnostics

Import libraries and functions

In [1]:
import time
from codebase_0 import CANN, MLP, cann_training_loop, mlp_train_loop, DeviceLoader, set_seed
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from torch import nn
import torch
import pyro

from poisson_bcann_vi import (
    build_bcann_guide, PoissonBCANN, poisson_fit_glm_mle,
    poisson_bcann_svi_training_loop, PoissonCANN, poisson_cann_training_loop,
    poisson_deviance, evaluate_bcann
)

from poisson_bcann_hmc_nuts import (
    nuts_diagnostics, predict_from_samples, nuts_posterior_stats, deviance_from_stats,
    svi_posterior_stats
)

from bcann_neutra import (
    neutra_diagnostics, neutra_posterior_stats, neutra_deviance_from_stats
)

from poisson_horseshoe import (
     HorseshoePoissonBCANN
)

seed = 42
set_seed(seed)
pyro.clear_param_store()

/Users/thangmai/Downloads/UNIMELB/HONOURS RESEARCH (ACTL40001)/.venv-honours/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load data and perform train/validation/test split

In [5]:
device = torch.device("cpu")

torch.set_default_device("cpu")


dataset = pd.read_csv("freMTPL2freq_transformed.csv", index_col= 0) 
y = torch.tensor(dataset["ClaimNb"].to_numpy(), dtype=torch.float32)
exposure = torch.tensor(dataset["Exposure"].to_numpy(), dtype=torch.float32)
X = torch.tensor(dataset.drop(columns=["IDpol", "Exposure","ClaimNb"]).to_numpy(), dtype=torch.float32)   # 48 columns

# Learning-test set split
X_learn, X_test, y_learn, y_test, exp_learn, exp_test = train_test_split(
    X, y, exposure, test_size= 0.1, random_state= seed
)

# Training-CV set split
X_train, X_val, y_train, y_val, exp_train, exp_val = train_test_split(
    X_learn, y_learn, exp_learn, test_size= 0.1, random_state= seed
)

learn_scaler = MinMaxScaler().fit(X_learn)
X_learn = learn_scaler.transform(X_learn)
X_test = learn_scaler.transform(X_test)

train_scaler = MinMaxScaler().fit(X_train)
X_train = train_scaler.transform(X_train)
X_val = train_scaler.transform(X_val)

def T(a):
    return torch.as_tensor(a, dtype=torch.float32, device=device)

X_train, X_val, X_learn, X_test = map(T, (X_train, X_val, X_learn, X_test))
y_train, y_val, y_learn, y_test = map(T, (y_train, y_val, y_learn, y_test))
exp_train, exp_val, exp_learn, exp_test = map(T, (exp_train, exp_val, exp_learn, exp_test))

torch.set_default_device(device)
train_loader = DeviceLoader(
        X_train, y_train, torch.log(exp_train),
        batch_size = 10000,
        shuffle= True
)
val_loader = DeviceLoader(
        X_val, y_val, torch.log(exp_val),
        batch_size= 10000
        
)
learn_loader = DeviceLoader(
        X_learn, y_learn, torch.log(exp_learn),
        batch_size= 10000
)
test_loader = DeviceLoader(
        X_test, y_test, torch.log(exp_test),
        batch_size= 10000
)

# NUTS diagnostics

Start a PoissonBCANN model

In [21]:
network_architecture = [20,15, 10]
poisson_bcann = PoissonBCANN(
    in_features= 48,
    neural_net_params={"hidden_activation_nodes": network_architecture,
                        "hidden_activations": nn.Tanh}
)

Load NUTS samples from disk and run diagnostics

In [13]:
nuts_run = torch.load("hmc_results/poisson_hmcnuts_full_set.pt", map_location="cpu")

nuts_samples, nuts_info = nuts_run["samples"], nuts_run["info"]        # [1, draws, ...]: one chain per file


In [14]:
nuts_rows = nuts_diagnostics(
    model=poisson_bcann, samples=nuts_samples, info=nuts_info,
    X_eval=X_learn, log_exposure_eval= torch.log(exp_learn), y_eval=y_learn,
    n_points=500, seed=seed, device=device,
)

Device used             : cuda:0
Wall time per chain (s) : [8689.3, 8826.3, 8801.5, 8818.7, 8620.5, 8790.0]
Divergences per chain   : [0, 0, 0, 0, 0, 0]
Adapted step size       : [0.03554, 0.03781, 0.03598, 0.04613, 0.04479, 0.03626]
quantity                                 max R-hat   min ESS
mu                                           1.000      9559
eta_glm                                      1.000      6151
f_nn                                         1.000      6307
cann.linear_pred.layer.bias                  1.000      7094
cann.linear_pred.layer.weight                1.000     10296
loglik (eval subset)                         1.000     10188


Making predictions

In [24]:
nuts_stats = nuts_posterior_stats(poisson_bcann, nuts_samples, X_test, log_exposure= torch.log(exp_test), batch = 500)

In [26]:
# mu_hat = nuts_stats["mu"]
print(deviance_from_stats(nuts_stats, y_test, log_exposure= torch.log(exp_test)))

0.30898720026016235


# NeuTra diagnostics

Load NeuTra samples from disk and run diagnostics

In [39]:
neutra_run = torch.load("hmc_results/poisson_neutra_full_set.pt", map_location="cpu")

neutra_samples, neutra_info = neutra_run["samples"], neutra_run["info"]        # [1, draws, ...]: one chain per file


In [40]:
neutra_rows = neutra_diagnostics(
    model=poisson_bcann, samples=neutra_samples, info=neutra_info,
    X_eval=X_learn, aux_eval= torch.log(exp_learn), y_eval=y_learn,
    n_points=500, seed=seed, device=device,
)

Device used             : ['cuda:0', 'cuda:0', 'cuda:0', 'cuda:0', 'cuda:0', 'cuda:0']
Wall time per chain (s) : [6710.8, 7150.1, 8466.5, 7124.7, 8332.6, 6745.7]
Divergences per chain   : [0, 0, 0, 0, 0, 0]
Adapted step size       : [0.03108, 0.02499, 0.02168, 0.03102, 0.02212, 0.03031]
quantity                                 max R-hat   min ESS
mu                                           1.006      2007
eta_glm                                      1.006       622
f_nn                                         1.004       744
cann.linear_pred.layer.weight                1.008      1427
cann.linear_pred.layer.bias                  1.009       447
loglik (eval subset)                         1.000      9651
z (random coordinates)                       1.000      4163
|z|^2                                        1.011       429


Making prediction on the test set

In [36]:
neutra_stats = neutra_posterior_stats(poisson_bcann, neutra_samples, X_test, batch = 500)

In [37]:
print(neutra_deviance_from_stats(model = poisson_bcann, stats = neutra_stats, y = y_test, aux= torch.log(exp_test)))

0.30898839235305786


# Horseshoe NUTS diagnostics

Start a HorseshoePoissonBCANN model

In [10]:
horseshoe_model = HorseshoePoissonBCANN(
        in_features=48, 
        neural_net_params={
            "hidden_activation_nodes": network_architecture,
            "hidden_activations": nn.Tanh
        }
    )

Load NUTS samples with horseshoe prior from disk and perform diagnostics

In [ ]:
horseshoe_nuts_run = torch.load("hmc_results/poisson_horseshoe_hmc_results.pt", map_location="cpu")

horseshoe_nuts_samples, horseshoe_nuts_info = horseshoe_nuts_run["samples"], horseshoe_nuts_run["info"]        # [1, draws, ...]: one chain per file


In [18]:
rows = nuts_diagnostics(
    model=horseshoe_model, samples=horseshoe_nuts_samples, info=horseshoe_nuts_info,
    X_eval=X_learn, log_exposure_eval= torch.log(exp_learn), y_eval=y_learn,
    n_points=500, seed=seed, device=device,
)

Device used             : cuda:0
Wall time per chain (s) : [12400.1, 12268.4, 12470.5, 11727.9, 12153.1, 12248.6]
Divergences per chain   : [6, 19, 0, 306, 53, 9]
Adapted step size       : [0.00024, 0.00087, 0.00055, 0.00139, 0.00072, 0.00064]
quantity                                 max R-hat   min ESS
mu                                           3.272         3
eta_glm                                      7.067         3
f_nn                                         4.336         3
cann.linear_pred.layer.bias                  5.971         3
cann.linear_pred.layer.weight                6.492         3
loglik (eval subset)                         1.749         6


# Horseshoe NeuTra diagnostics

Load NeuTra samples with horseshoe prior from disk and run diagnostics

In [19]:
horseshoe_neutra_run = torch.load("hmc_results/poisson_horseshoe_neutra_full_set.pt", map_location="cpu")

horseshoe_neutra_samples, horseshoe_neutra_info = horseshoe_neutra_run["samples"], horseshoe_neutra_run["info"]        # [1, draws, ...]: one chain per file


In [20]:
neutra_rows = neutra_diagnostics(
    model=horseshoe_model, samples=horseshoe_neutra_samples, info=horseshoe_neutra_info,
    X_eval=X_learn, aux_eval= torch.log(exp_learn), y_eval=y_learn,
    n_points=500, seed=seed, device=device,
)

Device used             : ['cuda:1', 'cuda:1', 'cuda:1', 'cuda:1', 'cuda:1', 'cuda:1']
Wall time per chain (s) : [13698.8, 13862.3, 13653.5, 13980.0, 13634.3, 13575.9]
Divergences per chain   : [0, 34, 37, 0, 5, 4]
Adapted step size       : [0.00031, 0.00151, 0.00121, 0.00058, 0.00107, 0.00103]
quantity                                 max R-hat   min ESS
mu                                           2.521         4
eta_glm                                      6.958         3
f_nn                                         4.224         3
cann.linear_pred.layer.weight                7.675         3
cann.linear_pred.layer.bias                  3.977         3
loglik (eval subset)                         1.157        44
z (random coordinates)                       4.492         3
|z|^2                                        5.959         3
